In [1]:
import pandas as pd
import os
from IPython.display import display
import numpy as np

# --- CONFIGURAÇÕES ---
base_project_path = os.getcwd() 
print(f"Diretório base do script: {base_project_path}")

# Definição dos caminhos
argus_dir = os.path.join(base_project_path, 'logs')
zeek_dir = os.path.join(base_project_path, 'logs', 'logs')

# Definindo os rotulos para esta categoria (Normal = 0)
label_value = 1

print(f"--- Processando Categoria: Analysis ---")
print(f"Diretório Zeek: {zeek_dir}")
print(f"Diretório Argus: {argus_dir}")

# --- FUNÇÕES ---
def read_zeek_log(file_path):
    """Lê um arquivo de log Zeek e retorna um DataFrame."""
    columns = []
    try:
        with open(file_path, 'r', encoding='latin-1') as f:
            for line in f:
                if line.startswith('#fields'):
                    columns = line.strip().split('\t')[1:]
                    break
        if not columns:
            print(f"Aviso: Não foi encontrada a linha #fields em {file_path}")
            return pd.DataFrame()

        df = pd.read_csv(
            file_path,
            sep='\t',
            names=columns,
            comment='#',
            header=None,
            low_memory=False,
            na_values=['-', '(empty)'],
            encoding='latin-1'
        )
        return df
    except Exception as e:
        print(f"Erro ao ler {file_path}: {e}")
        return pd.DataFrame()

def load_and_prepare_argus(argus_path):
    """Carrega CSV do Argus e padroniza para merge com Zeek."""
    if not os.path.exists(argus_path):
        print(f"Arquivo Argus não encontrado em {argus_path}")
        return None
    
    print(f"Carregando dados do Argus: {argus_path}")
    df_argus = pd.read_csv(argus_path, low_memory=False)
    
    # Limpeza dos nomes das colunas
    df_argus.columns = df_argus.columns.str.strip()

    # Mapeamento Argus -> Zeek/UNSW
    rename_map = {
        'SrcAddr': 'id.orig_h',
        'DstAddr': 'id.resp_h',
        'Sport':   'id.orig_p',
        'Dport':   'id.resp_p',
        'Proto':   'proto',
        'StartTime': 'argus_start',
        'LastTime':  'argus_end',
        'State':   'state',
        'Dur':     'dur',
        'sTtl':    'sttl',
        'dTtl':    'dttl',
        'SrcLoss': 'sloss',
        'DstLoss': 'dloss',
        'SrcPkts': 'spkts',
        'DstPkts': 'dpkts',
        'SrcWin':  'swin',
        'DstWin':  'dwin',
        'SrcTCPBase': 'stcpb',
        'DstTCPBase': 'dtcpb',
        'sMeanPktSz': 'smeansz',
        'dMeanPktSz': 'dmeansz',
        'SrcJitter':  'sjit',
        'DstJitter':  'djit',
        'SIntPkt':    'sinpkt',
        'DIntPkt':    'dinpkt',
        'TcpRtt':     'tcprtt',
        'SynAck':     'synack',
        'AckDat':     'ackdat',
        'SrcBytes':   'sbytes',
        'DstBytes':   'dbytes'
    }
    
    # Renomeia apenas o que existe
    actual_rename = {k: v for k, v in rename_map.items() if k in df_argus.columns}
    df_argus.rename(columns=actual_rename, inplace=True)

    for col in ['argus_start', 'argus_end']:
        if col in df_argus.columns:
            # tenta converter: se já vier numérico epoch, ok; se vier string, tenta parse
            df_argus[col] = pd.to_numeric(df_argus[col], errors='coerce')
            if df_argus[col].isna().mean() > 0.5:
                # provavelmente era string de data; tenta parse
                df_argus[col] = pd.to_datetime(df_argus[col], errors='coerce', utc=True).astype('int64') / 1e9
        # Se só tiver start, cria end como start (fluxos curtos)
        if 'argus_start' in df_argus.columns and 'argus_end' not in df_argus.columns:
            df_argus['argus_end'] = df_argus['argus_start']
        
        # Verificação de Segurança (Chaves de Merge)
        required_keys = ['id.orig_h', 'id.resp_h', 'id.orig_p', 'id.resp_p', 'proto', 'argus_start', 'argus_end']
        missing = [k for k in required_keys if k not in df_argus.columns]
        if missing:
            print(f"Colunas chave faltando no Argus após rename: {missing}")
            return None
        return df_argus

# --- EXECUÇÃO PRINCIPAL ---
log_data = {} 
log_types_info = {
    'conn': {'required': True, 'columns': None},
    'http': {'required': False, 'columns': ['uid', 'trans_depth', 'response_body_len', 'method']},
    'ftp': {'required': False, 'columns': ['uid', 'user', 'password', 'command']},
        'smtp': {'required': False, 'columns': ['uid', 'trans_depth']},
    'dns': {'required': False, 'columns': ['uid', 'query']}
}

if not os.path.isdir(zeek_dir):
    print(f"Diretório de logs Zeek não encontrado - {zeek_dir}")
else:
    all_files_in_dir = os.listdir(zeek_dir)

    # Leitura dos Logs Zeek (Na pasta ./logs/logs)
    for log_type, info in log_types_info.items():
        relevant_files = [f for f in all_files_in_dir if f.startswith(log_type + '.') and f.endswith('.log')]

        if not relevant_files:
            if info['required']:
                print(f"ERRO CRÍTICO: {log_type}.log necessário não encontrado em {zeek_dir}.")
            log_data[log_type] = pd.DataFrame()
            continue

        df_list = [read_zeek_log(os.path.join(zeek_dir, f)) for f in relevant_files]
        combined_df = pd.concat(df_list, ignore_index=True)

        if info['columns']:
            cols_to_keep = [col for col in info['columns'] if col in combined_df.columns]
            if 'uid' not in cols_to_keep and 'uid' in combined_df.columns:
                 cols_to_keep.insert(0, 'uid')
            if cols_to_keep:
                 combined_df = combined_df[cols_to_keep]

        log_data[log_type] = combined_df
        print(f"Lidos {len(relevant_files)} logs de {log_type}. Total linhas: {len(combined_df)}")


    # Junção dos Logs Zeek (Merge por UID)
    if 'conn' in log_data and not log_data['conn'].empty:
        final_df = log_data['conn'].copy()
        
        for log_type in ['http', 'ftp', 'smtp', 'dns']:
            if log_type in log_data and not log_data[log_type].empty and 'uid' in log_data[log_type].columns:
                log_data[log_type] = log_data[log_type].drop_duplicates(subset=['uid'], keep='first')
                cols_to_rename = {col: f"{log_type}_{col}" for col in log_data[log_type].columns if col != 'uid'}
                df_to_merge = log_data[log_type].rename(columns=cols_to_rename)
                final_df = pd.merge(final_df, df_to_merge, on='uid', how='left')

        
        # 2Ajuste de SERVICE/TRANS_DEPTH para SMTP (quando existir smtp.log)
        # Se a conexão foi analisada como SMTP, Zeek já costuma preencher 'service' = 'smtp'.
        # Caso não esteja preenchido, mas exista smtp_trans_depth, marcamos como smtp.
        if 'smtp_trans_depth' in final_df.columns:
            if 'service' not in final_df.columns:
                final_df['service'] = '-'
            # marca SMTP quando houver transação SMTP
            final_df.loc[final_df['smtp_trans_depth'].notna(), 'service'] = final_df.loc[final_df['smtp_trans_depth'].notna(), 'service'].replace('-', 'smtp')
            # fallback por porta (25)
            if 'id.resp_p' in final_df.columns:
                final_df.loc[(final_df['id.resp_p'] == 25) & (final_df['service'].isin(['-', ''])) , 'service'] = 'smtp'
        # Junção com Argus (Múltiplos CSVs)
        
        # Lista de arquivos para carregar (alterar o nome do csv para corresponder as capturas atuais)
        argus_filename_list = [
            "captura_analysis_kube1.csv"
        ]
        
        argus_dfs_list = []
        
        print("\nIniciando carregamento dos arquivos Argus...")
        for fname in argus_filename_list:
            full_argus_path = os.path.join(argus_dir, fname)
            df_temp = load_and_prepare_argus(full_argus_path)
            if df_temp is not None:
                argus_dfs_list.append(df_temp)
        
        if argus_dfs_list:
            # Concatena todos os DataFrames do Argus em um único Grande DataFrame
            df_argus_data = pd.concat(argus_dfs_list, ignore_index=True)
            
            # --- Correção de merge com padronização de chaves ---
            print("\nPadronizando chaves de merge zeek <-> argus...")
            
            # Função auxiliar para garantir inteiros em portas (trata 80.0 e 80)
            def clean_port(val):
                try:
                    return int(float(val))
                except:
                    return 0

            # Limpeza no Zeek (final_df)
            final_df['id.orig_p'] = final_df['id.orig_p'].apply(clean_port)
            final_df['id.resp_p'] = final_df['id.resp_p'].apply(clean_port)
            final_df['proto'] = final_df['proto'].astype(str).str.lower().str.strip()
            final_df['id.orig_h'] = final_df['id.orig_h'].astype(str).str.strip()
            final_df['id.resp_h'] = final_df['id.resp_h'].astype(str).str.strip()

            # Limpeza no Argus (df_argus_data)
            df_argus_data['id.orig_p'] = df_argus_data['id.orig_p'].apply(clean_port)
            df_argus_data['id.resp_p'] = df_argus_data['id.resp_p'].apply(clean_port)
            df_argus_data['proto'] = df_argus_data['proto'].astype(str).str.lower().str.strip()
            df_argus_data['id.orig_h'] = df_argus_data['id.orig_h'].astype(str).str.strip()
            df_argus_data['id.resp_h'] = df_argus_data['id.resp_h'].astype(str).str.strip()

            # --- MERGE ZEek + ARGUS por 5-tupla + aproximação temporal ---
            print("\nExecutando merge Zeek + Argus por 5-tupla + tempo (merge_asof)...")
            
            # Normaliza ts do Zeek (conn.log) para epoch float
            final_df['ts'] = pd.to_numeric(final_df['ts'], errors='coerce')
            
            # Gera chave 5-tupla
            def tuple_key(df, oh='id.orig_h', op='id.orig_p', rh='id.resp_h', rp='id.resp_p', pr='proto'):
                return (df[oh].astype(str) + "|" + df[op].astype(str) + "|" +
                        df[rh].astype(str) + "|" + df[rp].astype(str) + "|" +
                        df[pr].astype(str))
            
            final_df['_k5'] = tuple_key(final_df)
            df_argus_data['_k5'] = tuple_key(df_argus_data)
            
            # Escolhe o tempo de referência do Argus para o asof (meio do fluxo)
            df_argus_data['argus_mid'] = (df_argus_data['argus_start'] + df_argus_data['argus_end']) / 2.0
            
            # Ordena para merge_asof
            final_df = final_df.sort_values(['_k5', 'ts']).reset_index(drop=True)
            df_argus_data = df_argus_data.sort_values(['_k5', 'argus_mid']).reset_index(drop=True)
            
            # Faz merge_asof dentro de cada 5-tupla (por grupo)
            TOL_SECONDS = 2.0  # possivel ajuste aqui (1 a 3 segs em geral)
            
            merged_parts = []
            for k, zgrp in final_df.groupby('_k5', sort=False):
                agrp = df_argus_data[df_argus_data['_k5'] == k]
                if agrp.empty:
                    merged_parts.append(zgrp)
                    continue
            
                m = pd.merge_asof(
                    zgrp,
                    agrp,
                    left_on='ts',
                    right_on='argus_mid',
                    direction='nearest',
                    tolerance=TOL_SECONDS,
                    suffixes=('', '_argus')
                )
                merged_parts.append(m)
            
            final_df = pd.concat(merged_parts, ignore_index=True)
            
            # Filtro de validade: ts deve cair dentro do intervalo do fluxo argus (com tolerância)
            if 'argus_start' in final_df.columns and 'argus_end' in final_df.columns:
                ok = (
                    final_df['argus_start'].notna() &
                    final_df['argus_end'].notna() &
                    (final_df['ts'] >= (final_df['argus_start'] - TOL_SECONDS)) &
                    (final_df['ts'] <= (final_df['argus_end'] + TOL_SECONDS))
                )
                # se não ok, invalida as colunas vindas do Argus (para não “colar” fluxo errado)
                argus_cols = [c for c in df_argus_data.columns if c not in ['_k5']]
                for c in argus_cols:
                    if c in final_df.columns:
                        final_df.loc[~ok, c] = np.nan
            
            print("Merge temporal concluído.")

            # --- BLOCO DE PRIORIDADE E CONSOLIDAÇÃO ---
            # Sobrescreve colunas base do Zeek com Argus (que tem sttl, etc.)
            consolidation_map = {
                'dur': 'duration',      # Argus 'dur' -> Zeek 'duration'
                'sbytes': 'orig_bytes', # Argus 'sbytes' -> Zeek 'orig_bytes'
                'dbytes': 'resp_bytes',
                'spkts': 'orig_pkts',
                'dpkts': 'resp_pkts',
                'state': 'state'   # Argus 'state' (CON, FIN)
            }
            
            print("Consolidando colunas base (Argus > Zeek)...")
            for argus_col, zeek_col in consolidation_map.items():
                if argus_col in final_df.columns:
                    if zeek_col in final_df.columns:
                        # Usa combine_first: Se Argus tem valor, usa Argus. Se NaN, mantém Zeek
                        # Mas para forcar o Argus se disponível (pois Zeek não tem TTL)
                        # inverte ou usa fillna 
                        # Aqui usa o Argus para preencher buracos e sobrescrever se necessário
                        final_df[zeek_col] = final_df[argus_col].fillna(final_df[zeek_col])
                        
                        # Remove a coluna duplicada _argus ou original argus para limpar
                        final_df.drop(columns=[argus_col], inplace=True)
                    else:
                        # Se Zeek não tem, renomeia Argus para o nome esperado
                        final_df.rename(columns={argus_col: zeek_col}, inplace=True)
            
            # Garante preenchimento de zeros nas features exclusivas do Argus que não deram match
            new_features = ['sjit', 'djit', 'sinpkt', 'dinpkt', 'tcprtt', 'synack', 'ackdat', 'stcpb', 'dtcpb', 'swin', 'dwin', 'sttl', 'dttl', 'sloss', 'dloss']
            for feat in new_features:
                if feat in final_df.columns:
                    # Para features onde 0 pode ser "missing mascarado" (ex TTL/RTT/jitter), use -1
                    missing_sentinel = -1
                    
                    sentinel_feats = ['sjit', 'djit', 'sinpkt', 'dinpkt', 'tcprtt', 'synack', 'ackdat', 
                                      'stcpb', 'dtcpb', 'swin', 'dwin', 'sttl', 'dttl']
                    
                    zero_ok_feats = ['sloss', 'dloss']  # perdas podem ser 0 real com significado
                    
                    for feat in new_features:
                        if feat in final_df.columns:
                            if feat in sentinel_feats:
                                final_df[feat] = final_df[feat].fillna(missing_sentinel)
                            else:
                                final_df[feat] = final_df[feat].fillna(0)

        else:
             print("AVISO: Nenhum arquivo Argus foi carregado com sucesso. Features complexas serão 0.")

        # Rótulos e Exportação
        final_df['attack_cat'] = "Analysis"
        final_df['label'] = label_value
        
        # Limpeza de colunas temporárias
        cols_to_drop = [c for c in final_df.columns if c.endswith('_argus')]
        final_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

        print("\n--- Amostra do DataFrame Final (Verifique colunas sttl/dttl) ---")
        # Mostra colunas críticas para verificar se o merge funcionou
        cols_preview = ['ts', 'uid', 'id.orig_h', 'sttl', 'dttl', 'rate']
        cols_exist = [c for c in cols_preview if c in final_df.columns]
        display(final_df[cols_exist].head())

    else:
        print("ERRO: conn.log vazio ou não encontrado.")

Diretório base do script: C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\analysis
--- Processando Categoria: Analysis ---
Diretório Zeek: C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\analysis\logs\logs
Diretório Argus: C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\analysis\logs
Lidos 3 logs de conn. Total linhas: 2818
Lidos 2 logs de http. Total linhas: 3152
Lidos 2 logs de ftp. Total linhas: 1598
Lidos 1 logs de smtp. Total linhas: 3
Lidos 2 logs de dns. Total linhas: 120

Iniciando carregamento dos arquivos Argus...
Carregando dados do Argus: C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\ataque\analysis\logs\captura_analysis_kube1.csv

Padronizando chaves de merge zeek <-> argus...

Executando merge Zeek + Argus por 5-tupla + tempo (merge_asof)...
Merge temporal concluído.
Consolidando colunas base (Argus > Zeek)...

--- Amostra do DataFrame

,ts,uid,id.orig_h,sttl,dttl
0,1.773949e+09,Cepooy3k8cRIkvZQH7,NaN,-1.0,-1.0
1,1.773949e+09,CtWXr32DCqdUiH1emh,NaN,-1.0,-1.0
2,1.773951e+09,CK42eMNFkTCnM6Gpi,NaN,-1.0,-1.0
3,1.773949e+09,CnTGTz4ekgMC40K2Z1,NaN,-1.0,-1.0
4,1.773949e+09,C2RzIy4cBH3F10Vwg4,NaN,-1.0,-1.0


In [2]:
# --- ENGENHARIA DE FEATURES (CÁLCULOS SIMPLES) ---
# Assume que 'final_df' existe da célula anterior

print("\n--- Iniciando Engenharia de Features Simples ---")

# Verificar se o DataFrame base existe e não está vazio
if 'final_df' in locals() and not final_df.empty:

    # Tratar valores numéricos que podem ser string ou NaN antes dos cálculos
    numeric_cols_to_clean = ['duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts']
    for col in numeric_cols_to_clean:
        if col in final_df.columns:
            # Converte para numérico, erros viram NaN. Preenche NaN com 0.
            final_df[col] = pd.to_numeric(final_df[col], errors='coerce').fillna(0)
        else:
            print(f"Aviso: Coluna necessária '{col}' não encontrada para cálculos.")
            # Cria coluna com zeros se não existir para evitar erros posteriores
            final_df[col] = 0

    # Calcular 'rate'
    # Evita divisão por zero: np.divide(..., where=denominator!=0)
    total_pkts = final_df['orig_pkts'] + final_df['resp_pkts']
    final_df['rate'] = np.divide(total_pkts, final_df['duration'], \
                                 out=np.zeros_like(total_pkts, dtype=float), where=final_df['duration']!=0)

    # Calcular 'sload' (Source Load in bits per second)
    final_df['sload'] = np.divide(final_df['orig_bytes'] * 8, final_df['duration'], \
                                  out=np.zeros_like(final_df['orig_bytes'], dtype=float), where=final_df['duration']!=0)

    # Calcular 'dload' (Destination Load in bits per second)
    final_df['dload'] = np.divide(final_df['resp_bytes'] * 8, final_df['duration'], \
                                  out=np.zeros_like(final_df['resp_bytes'], dtype=float), where=final_df['duration']!=0)

    # Calcular 'smean' (Source Mean Packet Size)
    final_df['smean'] = np.divide(final_df['orig_bytes'], final_df['orig_pkts'], \
                                  out=np.zeros_like(final_df['orig_bytes'], dtype=float), where=final_df['orig_pkts']!=0).astype(int) # Usually integer

    # Calcular 'dmean' (Destination Mean Packet Size)
    final_df['dmean'] = np.divide(final_df['resp_bytes'], final_df['resp_pkts'], \
                                  out=np.zeros_like(final_df['resp_bytes'], dtype=float), where=final_df['resp_pkts']!=0).astype(int) # Usually integer

    # Calcular 'is_sm_ips_ports' (Source=Dest IP and Port)
    # Verifica se as colunas existem antes de comparar
    if 'id.orig_h' in final_df.columns and 'id.resp_h' in final_df.columns and \
       'id.orig_p' in final_df.columns and 'id.resp_p' in final_df.columns:
        final_df['is_sm_ips_ports'] = ((final_df['id.orig_h'] == final_df['id.resp_h']) & \
                                       (final_df['id.orig_p'] == final_df['id.resp_p'])).astype(int)
    else:
        print("Aviso: Colunas de IP/Porta não encontradas. 'is_sm_ips_ports' será 0.")
        final_df['is_sm_ips_ports'] = 0

    # Calcular 'is_ftp_login'
    # Verifica se as colunas do FTP (resultantes do merge) existem
    if 'ftp_user' in final_df.columns and 'ftp_password' in final_df.columns:
        # Será 1 se ambos user e password não forem NaN (ou seja, foram preenchidos no log)
        final_df['is_ftp_login'] = ((final_df['ftp_user'].notna()) & \
                                    (final_df['ftp_password'].notna())).astype(int)
    else:
        # Se não houve merge com ftp.log ou as colunas não existiam
        print("Aviso: Colunas 'ftp_user'/'ftp_password' não encontradas. 'is_ftp_login' será 0.")
        final_df['is_ftp_login'] = 0


    print("\n--- Amostra do DataFrame Após Adicionar Features Simples ---")
    display(final_df[['uid', 'duration', 'orig_pkts', 'resp_pkts', 'orig_bytes', 'resp_bytes', \
                      'rate', 'sload', 'dload', 'smean', 'dmean', 'is_sm_ips_ports', 'is_ftp_login', \
                      'attack_cat', 'label']].head()) # Mostra apenas algumas colunas chave + as novas
    

else:
    print("\nERRO: DataFrame 'final_df' não encontrado ou vazio. Execute a célula anterior primeiro.")


--- Iniciando Engenharia de Features Simples ---

--- Amostra do DataFrame Após Adicionar Features Simples ---


,uid,duration,orig_pkts,resp_pkts,orig_bytes,resp_bytes,rate,sload,dload,smean,dmean,is_sm_ips_ports,is_ftp_login,attack_cat,label
0,Cepooy3k8cRIkvZQH7,0.063437,3.0,0.0,0.0,0.0,47.291013,0.0,0.0,0,0,0,0,Analysis,1
1,CtWXr32DCqdUiH1emh,0.072757,2.0,0.0,0.0,0.0,27.488764,0.0,0.0,0,0,0,0,Analysis,1
2,CK42eMNFkTCnM6Gpi,0.024290,2.0,0.0,0.0,0.0,82.338411,0.0,0.0,0,0,0,0,Analysis,1
3,CnTGTz4ekgMC40K2Z1,0.000000,1.0,0.0,0.0,0.0,0.000000,0.0,0.0,0,0,0,0,Analysis,1
4,C2RzIy4cBH3F10Vwg4,0.000000,1.0,0.0,0.0,0.0,0.000000,0.0,0.0,0,0,0,0,Analysis,1


In [3]:
# --- ENGENHARIA DE FEATURES (AGREGAÇÕES ct_*) ---
# Assume que 'final_df' existe das células anteriores

print("\n--- Iniciando Engenharia de Features Agregadas (ct_*) ---")

# Verificar se o DataFrame base existe e não está vazio
if 'final_df' in locals() and not final_df.empty:

    # Garantir que os dados estejam ordenados por Timestamp
    print("Ordenando DataFrame por timestamp...")
    df_sorted = final_df.sort_values(by='ts').reset_index(drop=True)

    # Definir o tamanho da janela
    window_size = 100
    print(f"Usando uma janela deslizante de {window_size} conexões.")

    # Calcular as features ct_*

    # --- Features baseadas em IP/Porta/Serviço (Loop iterrows) ---
    results = {}
    print("Calculando features ct_* baseadas em IP/Porta/Serviço (pode levar algum tempo)...")
    if 'id.resp_h' in df_sorted.columns and 'id.resp_p' in df_sorted.columns:
        df_sorted['dst_ip_port'] = df_sorted['id.resp_h'].astype(str) + ':' + df_sorted['id.resp_p'].astype(str)
    if 'id.orig_h' in df_sorted.columns and 'id.orig_p' in df_sorted.columns:
        df_sorted['src_ip_port'] = df_sorted['id.orig_h'].astype(str) + ':' + df_sorted['id.orig_p'].astype(str)

    num_rows = len(df_sorted)
    
    for i, row in df_sorted.iterrows():
        start_idx = max(0, i - window_size + 1)
        current_window = df_sorted.iloc[start_idx : i + 1]

        # Calcula cada feature (lógica idêntica à anterior)
        if 'id.orig_h' in row and 'id.resp_p' in row and 'dst_ip_port' in row:
             results.setdefault('ct_srv_src', []).append(current_window[ (current_window['dst_ip_port'] == row['dst_ip_port']) & \
                                                                         (current_window['id.orig_h'] == row['id.orig_h']) ].shape[0])
        if 'id.resp_h' in row and 'id.orig_p' in row and 'src_ip_port' in row:
             results.setdefault('ct_srv_dst', []).append(current_window[ (current_window['src_ip_port'] == row['src_ip_port']) & \
                                                                         (current_window['id.resp_h'] == row['id.resp_h']) ].shape[0])
        if 'id.resp_h' in row:
             results.setdefault('ct_dst_ltm', []).append(current_window[ current_window['id.resp_h'] == row['id.resp_h'] ].shape[0])
        if 'id.orig_h' in row:
             results.setdefault('ct_src_ltm', []).append(current_window[ current_window['id.orig_h'] == row['id.orig_h'] ].shape[0])
        if 'id.orig_h' in row and 'id.resp_p' in row:
             results.setdefault('ct_src_dport_ltm', []).append(current_window[ (current_window['id.orig_h'] == row['id.orig_h']) & \
                                                                               (current_window['id.resp_p'] == row['id.resp_p']) ].shape[0])
        if 'id.resp_h' in row and 'id.orig_p' in row:
             results.setdefault('ct_dst_sport_ltm', []).append(current_window[ (current_window['id.resp_h'] == row['id.resp_h']) & \
                                                                               (current_window['id.orig_p'] == row['id.orig_p']) ].shape[0])
        if 'id.orig_h' in row and 'id.resp_h' in row:
             results.setdefault('ct_dst_src_ltm', []).append(current_window[ (current_window['id.orig_h'] == row['id.orig_h']) & \
                                                                             (current_window['id.resp_h'] == row['id.resp_h']) ].shape[0])

        if (i + 1) % 500 == 0:
            print(f"  Processado {i + 1}/{num_rows} linhas...")

    print("Cálculo das features baseadas em IP/Porta/Serviço concluído.")
    for feature_name, values in results.items():
         if len(values) == len(df_sorted):
              df_sorted[feature_name] = values
         else:
              print(f"Erro de tamanho: '{feature_name}'. Preenchendo com 0.")
              df_sorted[feature_name] = 0
    df_sorted = df_sorted.drop(columns=['dst_ip_port', 'src_ip_port'], errors='ignore')

    # -- Features Específicas (ct_state_ttl, ct_ftp_cmd, ct_flw_http_mthd) --

    # ct_state_ttl (Contagem por estado) - Usando Factorization e indexação NumPy
    print("Calculando ct_state_ttl (regra discreta baseada em state+sttl+dttl)...")
    
    # ct_state_ttl no UNSW é um código discreto por combinações específicas
    # (em muitos labs ele vira 0 mesmo; isso é esperado se os TTLs não batem nos valores do UNSW)
    
    if 'sttl' in df_sorted.columns and 'dttl' in df_sorted.columns and 'state' in df_sorted.columns:
        sttl = pd.to_numeric(df_sorted['sttl'], errors='coerce').fillna(-1).astype(int)
        dttl = pd.to_numeric(df_sorted['dttl'], errors='coerce').fillna(-1).astype(int)
        state = df_sorted['state'].fillna('').astype(str)
    
        ct = np.zeros(len(df_sorted), dtype=int)
    
        # 1) FIN + (sttl in {62,63,254,255}) + (dttl in {252,253})
        m = state.eq('FIN') & sttl.isin([62, 63, 254, 255]) & dttl.isin([252, 253])
        ct[m] = 1
    
        # 2) INT + (sttl in {0,62,254}) + (dttl==0)
        # no seu caso, 0 pode ser "missing"; se você já usa -1, mantenha a regra com 0 literal
        m = state.eq('INT') & sttl.isin([0, 62, 254]) & dttl.eq(0)
        ct[m] = 2
    
        # 3) CON + (sttl in {62,254}) + (dttl in {60,252,253})
        m = state.eq('CON') & sttl.isin([62, 254]) & dttl.isin([60, 252, 253])
        ct[m] = 3
    
        # 4) ACC + (sttl==254) + (dttl==252)
        m = state.eq('ACC') & sttl.eq(254) & dttl.eq(252)
        ct[m] = 4
    
        # 5) CLO + (sttl==254) + (dttl==252)
        m = state.eq('CLO') & sttl.eq(254) & dttl.eq(252)
        ct[m] = 5
    
        # 7) REQ + (sttl==254) + (dttl==0)
        m = state.eq('REQ') & sttl.eq(254) & dttl.eq(0)
        ct[m] = 7
    
        df_sorted['ct_state_ttl'] = ct
    else:
        print("Aviso: faltam colunas para ct_state_ttl. Definindo 0.")
        df_sorted['ct_state_ttl'] = 0

    # ct_ftp_cmd (Lógica idêntica à anterior)
    print("Calculando ct_ftp_cmd...")
    if 'ftp_command' in df_sorted.columns:
        df_sorted['ct_ftp_cmd'] = df_sorted['ftp_command'].notna().rolling(window=window_size, min_periods=1).sum().fillna(0).astype(int)
    else:
        print("Aviso: Coluna 'ftp_command' não encontrada. 'ct_ftp_cmd' será 0.")
        df_sorted['ct_ftp_cmd'] = 0

    # ct_flw_http_mthd (Lógica idêntica à anterior)
    print("Calculando ct_flw_http_mthd...")
    if 'http_method' in df_sorted.columns:
        df_sorted['ct_flw_http_mthd'] = df_sorted['http_method'].notna().rolling(window=window_size, min_periods=1).sum().fillna(0).astype(int)
    else:
        print("Aviso: Coluna 'http_method' não encontrada. 'ct_flw_http_mthd' será 0.")
        df_sorted['ct_flw_http_mthd'] = 0

    print("\n--- Amostra do DataFrame Após Adicionar Features Agregadas ---")
    ct_cols_to_show = [col for col in df_sorted.columns if col.startswith('ct_')]
    display(df_sorted[['ts', 'uid', 'id.orig_h', 'id.resp_h'] + ct_cols_to_show].head())
    display(df_sorted[['ts', 'uid', 'id.orig_h', 'id.resp_h'] + ct_cols_to_show].tail())

    final_engineered_df = df_sorted # Atribui o resultado final
    # --- Exportar para CSV (Opcional) ---
    output_filename = f"{"analysis".lower().replace(os.path.sep, '_')}_processed.csv" # Nomeia arquivo baseado na categoria
    final_engineered_df.to_csv(os.path.join(base_project_path, output_filename), index=False)
    print(f"\nDataFrame exportado para {output_filename}")

else:
    print("\nERRO: DataFrame 'final_df' não encontrado ou vazio. Execute as células anteriores primeiro.")


--- Iniciando Engenharia de Features Agregadas (ct_*) ---
Ordenando DataFrame por timestamp...
Usando uma janela deslizante de 100 conexões.
Calculando features ct_* baseadas em IP/Porta/Serviço (pode levar algum tempo)...
  Processado 500/2818 linhas...
  Processado 1000/2818 linhas...
  Processado 1500/2818 linhas...
  Processado 2000/2818 linhas...
  Processado 2500/2818 linhas...
Cálculo das features baseadas em IP/Porta/Serviço concluído.
Calculando ct_state_ttl (regra discreta baseada em state+sttl+dttl)...
Aviso: faltam colunas para ct_state_ttl. Definindo 0.
Calculando ct_ftp_cmd...
Calculando ct_flw_http_mthd...

--- Amostra do DataFrame Após Adicionar Features Agregadas ---


,ts,uid,id.orig_h,id.resp_h,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_state_ttl,ct_ftp_cmd,ct_flw_http_mthd
0,1.773947e+09,C2PGEF2FHs1vP1l0kk,NaN,NaN,0,0,0,0,0,0,0,0,0,0
1,1.773947e+09,CEu6lQ2i5UxjDvRm6d,NaN,NaN,0,0,0,0,0,0,0,0,0,0
2,1.773947e+09,C03igKMG82AcwKt9b,NaN,NaN,0,0,0,0,0,0,0,0,1,0
3,1.773947e+09,CSiiDPVgIv4NrdAke,NaN,NaN,0,0,0,0,0,0,0,0,1,0
4,1.773947e+09,CXJfW91Bu12WTfPobl,NaN,NaN,0,0,0,0,0,0,0,0,1,1


,ts,uid,id.orig_h,id.resp_h,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_state_ttl,ct_ftp_cmd,ct_flw_http_mthd
2813,1.773953e+09,CZHAaS7996CrrDb18,NaN,NaN,0,0,0,0,0,0,0,0,16,27
2814,1.773953e+09,CFF36K2ku1S50GwZ0c,NaN,NaN,0,0,0,0,0,0,0,0,16,27
2815,1.773953e+09,CvHPA03Qe41JutqELi,NaN,NaN,0,0,0,0,0,0,0,0,16,27
2816,1.773953e+09,CRFrE03UL1yQ4p7Fae,NaN,NaN,0,0,0,0,0,0,0,0,16,27
2817,1.773953e+09,Cgl6Sr2w5HeWTF0Zdj,NaN,NaN,0,0,0,0,0,0,0,0,16,26



DataFrame exportado para analysis_processed.csv
